# Collateral Requirements Engine - Step by Step Testing

This notebook walks through testing the configurable collateral requirements engine with small, focused steps.

## Step 1: Setup and Imports

In [1]:
import sys
import os
from pathlib import Path

print("Setting up paths...")
print(f"Current working directory: {os.getcwd()}")

Setting up paths...
Current working directory: /home/mpo/algorand-showcase/algorand-lending-ecosystem/business-logic-engines/blockchain-collateral-analyzer/collateral-requirements/notebooks


In [2]:
# Add the parent directory to path so we can import our modules
parent_dir = Path(os.getcwd()).parent
sys.path.insert(0, str(parent_dir))

print(f"Added to path: {parent_dir}")
print("Path setup complete!")

Added to path: /home/mpo/algorand-showcase/algorand-lending-ecosystem/business-logic-engines/blockchain-collateral-analyzer/collateral-requirements
Path setup complete!


In [3]:
# Import the required modules
try:
    from core.collateral_engine import CollateralRequirementsEngine
    from core.config import load_config
    print("✅ Successfully imported collateral engine modules")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please make sure you're running this from the notebooks directory")

✅ Successfully imported collateral engine modules


## Step 2: Load Configuration

In [4]:
# Load the configuration
try:
    config = load_config()
    print("✅ Configuration loaded successfully!")
except Exception as e:
    print(f"❌ Error loading config: {e}")

✅ Configuration loaded successfully!


In [5]:
# Display basic configuration info
print("📋 Configuration Summary:")
print(f"  MCP Reader URL: {config.mcp_services.algorand_reader_url}")
print(f"  MCP Market URL: {config.mcp_services.market_data_url}")
print(f"  Database path: {config.database.default_path}")

📋 Configuration Summary:
  MCP Reader URL: http://localhost:8002
  MCP Market URL: http://localhost:8003
  Database path: notebooks/collateral_analysis.db


In [6]:
# Show collateral ratios
print("💰 Base Collateral Ratios:")
print(f"  ALGO Native: {config.base_collateral_ratios.algo_native:.2f}x")
print(f"  Stablecoin: {config.base_collateral_ratios.stablecoin:.2f}x")
print(f"  ASA Token: {config.base_collateral_ratios.asa_token:.2f}x")
print(f"  LP Token: {config.base_collateral_ratios.lp_token:.2f}x")

💰 Base Collateral Ratios:
  ALGO Native: 1.50x
  Stablecoin: 1.10x
  ASA Token: 2.00x
  LP Token: 3.00x


## Step 3: Initialize Engine

In [7]:
# Initialize the collateral engine
try:
    engine = CollateralRequirementsEngine()
    print("✅ Engine initialized successfully!")
except Exception as e:
    print(f"❌ Error initializing engine: {e}")

✅ Engine initialized successfully!


In [8]:
# Check engine configuration
print("🚀 Engine Configuration:")
print(f"  Database path: {engine.db.db_path}")
print(f"  MCP Reader URL: {engine.mcp_reader_url}")
print(f"  MCP Market URL: {engine.mcp_market_url}")

🚀 Engine Configuration:
  Database path: /home/mpo/algorand-showcase/algorand-lending-ecosystem/business-logic-engines/blockchain-collateral-analyzer/collateral-requirements/notebooks/collateral_analysis.db
  MCP Reader URL: http://localhost:8002
  MCP Market URL: http://localhost:8003


In [9]:
# Check database statistics
try:
    stats = engine.db.get_analysis_statistics()
    print("📊 Database Statistics:")
    print(f"  Total analyses: {stats['total_analyses']}")
    print(f"  Analyses last 24h: {stats['analyses_last_24h']}")
    if stats['average_collateral_ratio']:
        print(f"  Average ratio: {stats['average_collateral_ratio']:.2f}x")
except Exception as e:
    print(f"❌ Database error: {e}")

📊 Database Statistics:
  Total analyses: 36
  Analyses last 24h: 36
  Average ratio: 2.21x


## Step 4: Simple ALGO Analysis

In [10]:
# Define a simple test scenario
loan_amount = 10000  # $10k loan
algo_amount = 50000  # 50k ALGO tokens

print("🔍 Test Scenario:")
print(f"  Loan Amount: ${loan_amount:,}")
print(f"  Collateral: {algo_amount:,} ALGO")

🔍 Test Scenario:
  Loan Amount: $10,000
  Collateral: 50,000 ALGO


In [11]:
# Create collateral positions
collateral_positions = [{
    'asset_id': '0',
    'asset_symbol': 'ALGO',
    'amount': algo_amount
}]

print("✅ Collateral positions created")
for pos in collateral_positions:
    print(f"  - {pos['amount']:,} {pos['asset_symbol']} (ID: {pos['asset_id']})")

✅ Collateral positions created
  - 50,000 ALGO (ID: 0)


In [12]:
# Run the analysis
import asyncio

print("⏳ Running collateral analysis...")

try:
    result = await engine.analyze_collateral_requirement(
        loan_amount_usd=loan_amount,
        collateral_positions=collateral_positions,
        loan_id="NOTEBOOK-TEST-001",
        market_conditions="normal"
    )
    print("✅ Analysis completed successfully!")
except Exception as e:
    print(f"❌ Analysis failed: {e}")
    import traceback
    traceback.print_exc()

⏳ Running collateral analysis...
✅ Analysis completed successfully!


## Step 5: Display Results

In [13]:
# Show key metrics
if 'result' in locals():
    print("🎯 Analysis Results:")
    print(f"  Required Ratio: {result.required_collateral_ratio:.2f}x")
    print(f"  Risk Level: {result.risk_level}")
    print(f"  Confidence: {result.confidence_score:.1%}")
else:
    print("❌ No results available - analysis may have failed")

🎯 Analysis Results:
  Required Ratio: 2.15x
  Risk Level: critical
  Confidence: 68.0%


In [14]:
# Show collateral values
if 'result' in locals():
    required_value = result.loan_amount_usd * result.required_collateral_ratio
    current_value = result.total_collateral_value_usd
    shortage = required_value - current_value
    
    print("💰 Collateral Values:")
    print(f"  Current Value: ${current_value:,.2f}")
    print(f"  Required Value: ${required_value:,.2f}")
    
    if shortage > 0:
        print(f"  ⚠️ Shortage: ${shortage:,.2f}")
    else:
        print(f"  ✅ Excess: ${-shortage:,.2f}")

💰 Collateral Values:
  Current Value: $11,683.70
  Required Value: $21,450.00
  ⚠️ Shortage: $9,766.30


In [15]:
# Show position details
if 'result' in locals():
    print("📋 Position Details:")
    for pos in result.collateral_positions:
        print(f"  {pos.asset_symbol}: {pos.position_size:,.0f} @ ${pos.current_price_usd:.4f}")
        print(f"    Total Value: ${pos.position_value_usd:,.2f}")
        print(f"    Asset Type: {pos.asset_type.value}")

📋 Position Details:
  ALGO: 50,000 @ $0.2337
    Total Value: $11,683.70
    Asset Type: algo_native


## Step 6: Show Recommendations

In [16]:
# Display recommendations
if 'result' in locals() and result.recommendations:
    print("💡 Recommendations:")
    for i, rec in enumerate(result.recommendations, 1):
        print(f"  {i}. {rec}")
else:
    print("No recommendations available")

💡 Recommendations:
  1. Add $9,766 more collateral to meet requirements
  2. Consider reducing loan amount or adding more stable collateral
  3. Improve diversification by adding different asset types


## Step 7: Liquidation Scenarios

In [17]:
# Show liquidation scenarios
if 'result' in locals() and result.liquidation_scenarios:
    print("⚠️ Liquidation Scenarios:")
    for scenario in result.liquidation_scenarios:
        drop_pct = scenario['price_drop_percentage'] * 100
        recovery_pct = scenario['net_recovery_percentage'] * 100
        prob_pct = scenario['scenario_probability'] * 100
        
        print(f"  {scenario['scenario_name']}:")
        print(f"    Price Drop: {drop_pct:.0f}%")
        print(f"    Recovery: {recovery_pct:.1f}%")
        print(f"    Probability: {prob_pct:.1f}%")
        print()

⚠️ Liquidation Scenarios:
  Minor Correction:
    Price Drop: 10%
    Recovery: 0.0%
    Probability: 25.0%

  Market Dip:
    Price Drop: 20%
    Recovery: 0.0%
    Probability: 20.0%

  Bear Market:
    Price Drop: 30%
    Recovery: 0.0%
    Probability: 15.0%

  Black Swan:
    Price Drop: 50%
    Recovery: 0.0%
    Probability: 5.0%



## Step 8: Test Different Market Conditions

In [18]:
# Test different market conditions
market_conditions = ['bull', 'normal', 'bear', 'volatile']
market_results = {}

print("📊 Testing Market Conditions:")
for market in market_conditions:
    print(f"  Testing {market} market...")

📊 Testing Market Conditions:
  Testing bull market...
  Testing normal market...
  Testing bear market...
  Testing volatile market...


In [19]:
# Run analysis for each market condition
test_positions = [{
    'asset_id': '0',
    'asset_symbol': 'ALGO',
    'amount': 30000
}]

test_loan = 5000

for market in market_conditions:
    try:
        market_result = await engine.analyze_collateral_requirement(
            loan_amount_usd=test_loan,
            collateral_positions=test_positions,
            loan_id=f"MARKET-{market.upper()}",
            market_conditions=market
        )
        market_results[market] = market_result.required_collateral_ratio
        print(f"  ✅ {market.title()}: {market_result.required_collateral_ratio:.2f}x")
    except Exception as e:
        print(f"  ❌ {market.title()}: Failed - {e}")

  ✅ Bull: 1.93x
  ✅ Normal: 2.15x
  ✅ Bear: 2.47x
  ✅ Volatile: 2.68x


In [20]:
# Show market condition comparison
if market_results:
    min_ratio = min(market_results.values())
    max_ratio = max(market_results.values())
    difference = ((max_ratio / min_ratio) - 1) * 100
    
    print("📈 Market Condition Impact:")
    print(f"  Lowest ratio: {min_ratio:.2f}x")
    print(f"  Highest ratio: {max_ratio:.2f}x")
    print(f"  Difference: {difference:.1f}%")

📈 Market Condition Impact:
  Lowest ratio: 1.93x
  Highest ratio: 2.68x
  Difference: 38.8%


## Step 9: Database Check

In [21]:
# Check updated database statistics
try:
    updated_stats = engine.db.get_analysis_statistics()
    print("📊 Updated Database Statistics:")
    print(f"  Total analyses: {updated_stats['total_analyses']}")
    print(f"  Analyses last 24h: {updated_stats['analyses_last_24h']}")
    
    if updated_stats['average_collateral_ratio']:
        print(f"  Average ratio: {updated_stats['average_collateral_ratio']:.2f}x")
    
    if updated_stats['risk_level_distribution']:
        print("  Risk distribution:")
        for risk, count in updated_stats['risk_level_distribution'].items():
            print(f"    {risk}: {count}")
except Exception as e:
    print(f"❌ Database error: {e}")

📊 Updated Database Statistics:
  Total analyses: 41
  Analyses last 24h: 41
  Average ratio: 2.22x
  Risk distribution:
    critical: 30
    high: 8
    medium: 3


## Step 10: MCP Service Connectivity Test

In [22]:
# Test MCP service connectivity
import aiohttp

print("🔗 Testing MCP Service Connectivity:")

services = {
    'Algorand Reader': engine.mcp_reader_url,
    'Market Data': engine.mcp_market_url
}

print("Services to test:")
for name, url in services.items():
    print(f"  {name}: {url}")

🔗 Testing MCP Service Connectivity:
Services to test:
  Algorand Reader: http://localhost:8002
  Market Data: http://localhost:8003


In [23]:
# Perform connectivity tests
timeout = aiohttp.ClientTimeout(total=3)
mcp_services_available = False

async with aiohttp.ClientSession(timeout=timeout) as session:
    for name, url in services.items():
        try:
            async with session.get(f"{url}/health") as response:
                if response.status == 200:
                    print(f"  ✅ {name}: Connected")
                    mcp_services_available = True
                else:
                    print(f"  ⚠️ {name}: HTTP {response.status}")
        except aiohttp.ClientConnectorError:
            print(f"  ❌ {name}: Connection refused (service not running)")
        except Exception as e:
            print(f"  ❌ {name}: {str(e)}")

# Show instructions based on MCP service availability
if not mcp_services_available:
    print("\n💡 MCP Services Not Running:")
    print("  The engine is using fallback prices from configuration.")
    print("  To get live data, start MCP services with:")
    print("  \n  cd ../..")
    print("  ./start-mcp-services.sh")
    print("  \n  Then re-run this cell to test connectivity.")
else:
    print("\n🎉 MCP Services Available!")
    print("  The engine can now fetch real-time data from Algorand blockchain.")

  ✅ Algorand Reader: Connected
  ✅ Market Data: Connected

🎉 MCP Services Available!
  The engine can now fetch real-time data from Algorand blockchain.


## Summary

In [24]:
print("🎉 Collateral Engine Testing Complete!")
print("")
print("✅ What we tested:")
print("  • Configuration loading")
print("  • Engine initialization")
print("  • Basic ALGO analysis")
print("  • Market condition sensitivity")
print("  • Database storage")
print("  • MCP service connectivity")
print("")
print("📝 Next steps:")
print("  • Start MCP services for live data")
print("  • Try the CLI tool")
print("  • Customize configuration")
print("  • Integrate with lending platform")

🎉 Collateral Engine Testing Complete!

✅ What we tested:
  • Configuration loading
  • Engine initialization
  • Basic ALGO analysis
  • Market condition sensitivity
  • Database storage
  • MCP service connectivity

📝 Next steps:
  • Start MCP services for live data
  • Try the CLI tool
  • Customize configuration
  • Integrate with lending platform
